In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("cardiology_dataset.csv")

In [3]:
df.shape

(5100, 18)

In [4]:
df.head()

,Age,Gender,ChestPain,ShortnessBreath,Dizziness,Palpitations,Fatigue,Sweating,Nausea,BloodPressure,HeartRate,Cholesterol,DiabetesHistory,Smoking,Obesity,FamilyHistoryHeartDisease,Disease,Medicine
0,18,1,1,0,0,1,0,0,1,152,90,259,1,1,1,0,Hypertension,Amlodipine / Lisinopril
1,21,1,1,1,1,1,1,1,0,149,81,219,1,0,1,0,Angina,Nitroglycerin / Aspirin
2,71,1,1,1,1,1,1,1,1,152,100,226,1,1,1,1,Coronary Artery Disease,Aspirin / Statins
3,63,0,1,0,1,0,0,0,0,193,77,228,1,0,0,1,Hypertension,Amlodipine / Lisinopril
4,24,0,0,0,0,1,0,0,0,90,65,200,1,0,0,1,Normal,No medication; routine check-up


In [5]:
# Encode categorical target columns (Disease, Medicine)
disease_encoder = LabelEncoder()
medicine_encoder = LabelEncoder()

df["Disease_encoded"] = disease_encoder.fit_transform(df["Disease"])
df["Medicine_encoded"] = medicine_encoder.fit_transform(df["Medicine"])

In [6]:
X = df.drop(["Disease", "Medicine", "Disease_encoded", "Medicine_encoded"], axis=1)
y_disease = df["Disease_encoded"]
y_medicine = df["Medicine_encoded"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_disease, test_size=0.2, random_state=42, stratify=y_disease
)

In [8]:
model = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42)

In [9]:
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, n_estimators=150, random_state=42)

In [10]:
y_pred = model.predict(X_test)

In [11]:
accuracy = accuracy_score(y_test, y_pred)

In [12]:
print(f"Disease Prediction Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=disease_encoder.classes_))

Disease Prediction Accuracy: 90.88%

Classification Report:
                         precision    recall  f1-score   support

                 Angina       0.90      0.83      0.87       170
             Arrhythmia       0.99      1.00      0.99       170
Coronary Artery Disease       0.82      0.68      0.75       170
          Heart Disease       0.77      0.99      0.86       170
           Hypertension       1.00      0.95      0.98       170
                 Normal       1.00      1.00      1.00       170

               accuracy                           0.91      1020
              macro avg       0.91      0.91      0.91      1020
           weighted avg       0.91      0.91      0.91      1020



In [13]:
# Train a second model to predict the recommended Medicine
X_train_m, X_test_m, ym_train, ym_test = train_test_split(
    X, y_medicine, test_size=0.2, random_state=42, stratify=y_medicine
)

medicine_model = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42)
medicine_model.fit(X_train_m, ym_train)

ym_pred = medicine_model.predict(X_test_m)
medicine_accuracy = accuracy_score(ym_test, ym_pred)

print(f"Medicine Prediction Accuracy: {medicine_accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(ym_test, ym_pred, target_names=medicine_encoder.classes_))

Medicine Prediction Accuracy: 92.16%

Classification Report:
                                        precision    recall  f1-score   support

               Amlodipine / Lisinopril       1.00      0.95      0.97       170
                     Aspirin / Statins       0.84      0.77      0.80       170
Cardiologist consultation + medication       0.80      0.99      0.88       170
               Metoprolol / Amiodarone       1.00      1.00      1.00       170
               Nitroglycerin / Aspirin       0.92      0.82      0.87       170
       No medication; routine check-up       1.00      1.00      1.00       170

                              accuracy                           0.92      1020
                             macro avg       0.93      0.92      0.92      1020
                          weighted avg       0.93      0.92      0.92      1020



In [14]:
import joblib

joblib.dump(model, "heart_disease_model.pkl")
joblib.dump(medicine_model, "heart_medicine_model.pkl")
joblib.dump(disease_encoder, "heart_disease_encoder.pkl")
joblib.dump(medicine_encoder, "heart_medicine_encoder.pkl")

print("Heart Disease & Medicine Models Saved Successfully")

Heart Disease & Medicine Models Saved Successfully


In [15]:
# Example: predict disease + medicine for a new patient
sample = X_test.iloc[[0]]
pred_disease = disease_encoder.inverse_transform(model.predict(sample))
pred_medicine = medicine_encoder.inverse_transform(medicine_model.predict(sample))

print("Predicted Disease:", pred_disease[0])
print("Recommended Medicine:", pred_medicine[0])

Predicted Disease: Heart Disease
Recommended Medicine: Cardiologist consultation + medication
